# 02 — Experimentación del módulo CSP

Pruebas del solver `src/csp/solver.py` sobre perfiles de parcelas guatemaltecas. Valida que las restricciones duras (altitud, pH, sombra) filtran correctamente y compara el comportamiento con y sin diversificación inter-parcelas.

**Modelo del problema:**
- Variables: una por parcela (`P1..Pn`).
- Dominio inicial: variedades de `varieties.csv` que pasan las restricciones duras de cada parcela.
- Restricción inter-parcelas (opcional): AllDifferent (`diversify=True`) — diversifica variedades para mitigar riesgo sistémico de roya en la finca.
- Soft score: media ponderada de `rust_resistance` y `yield_score`, normalizada a `[0, 1]`.

In [1]:
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.csp.varieties_loader import load_varieties, satisfies_hard_constraints
from src.csp.solver import Parcela, solve

varieties = load_varieties('../data/raw/varieties.csv')
len(varieties), [v.name for v in varieties]

(12,
 ['Caturra',
  'Bourbon',
  'Catuai',
  'Typica',
  'Pache',
  'Pacas',
  'Pacamara',
  'Maragogype',
  'Gesha',
  'Catimor',
  'Anacafe 14',
  'Marsellesa'])

## 1. Casos de prueba

Seis parcelas con perfiles distintos cubriendo el rango altitudinal cafetalero de Guatemala (Acatenango ~1600m, Huehuetenango ~1700m, San Marcos ~1300m, Cobán bajío ~900m). Sombra: 0 sol pleno, 1 semi-sombra, 2 sombra densa.

In [2]:
parcelas = [
    Parcela('P1_Acatenango_alto',  altitud=1600, ph=5.4, sombra_disponible=1),
    Parcela('P2_Huehue_alto',      altitud=1750, ph=5.6, sombra_disponible=2),
    Parcela('P3_SanMarcos_medio',  altitud=1300, ph=5.2, sombra_disponible=1),
    Parcela('P4_Coban_bajo',       altitud=900,  ph=5.5, sombra_disponible=0),
    Parcela('P5_FraijanesAlto',    altitud=1800, ph=5.8, sombra_disponible=2),
    Parcela('P6_Antigua_medio',    altitud=1500, ph=5.0, sombra_disponible=1),
]
pd.DataFrame([p.__dict__ for p in parcelas])

,parcela_id,altitud,ph,sombra_disponible
0,P1_Acatenango_alto,1600,5.4,1
1,P2_Huehue_alto,1750,5.6,2
2,P3_SanMarcos_medio,1300,5.2,1
3,P4_Coban_bajo,900,5.5,0
4,P5_FraijanesAlto,1800,5.8,2
5,P6_Antigua_medio,1500,5.0,1


## 2. Dominios iniciales (variedades que pasan restricciones duras)

In [3]:
dominios = {
    p.parcela_id: [
        v.name for v in varieties
        if satisfies_hard_constraints(v, p.altitud, p.ph, p.sombra_disponible)
    ]
    for p in parcelas
}
for pid, doms in dominios.items():
    print(f'{pid:30s} ({len(doms)}) -> {doms}')

P1_Acatenango_alto             (9) -> ['Caturra', 'Bourbon', 'Catuai', 'Pache', 'Pacas', 'Pacamara', 'Maragogype', 'Anacafe 14', 'Marsellesa']
P2_Huehue_alto                 (5) -> ['Bourbon', 'Typica', 'Pacamara', 'Gesha', 'Anacafe 14']
P3_SanMarcos_medio             (10) -> ['Caturra', 'Bourbon', 'Catuai', 'Pache', 'Pacas', 'Pacamara', 'Maragogype', 'Catimor', 'Anacafe 14', 'Marsellesa']
P4_Coban_bajo                  (1) -> ['Catimor']
P5_FraijanesAlto               (5) -> ['Bourbon', 'Typica', 'Pacamara', 'Gesha', 'Anacafe 14']
P6_Antigua_medio               (9) -> ['Caturra', 'Bourbon', 'Catuai', 'Pache', 'Pacas', 'Pacamara', 'Maragogype', 'Anacafe 14', 'Marsellesa']


## 3. Resolución con diversificación (`diversify=True`)

AllDifferent activo: ninguna variedad se repite entre parcelas. Forward Checking poda dominios al asignar; MRV procesa primero la parcela con menos opciones.

In [4]:
t0 = time.perf_counter()
sol_div = solve(parcelas, varieties, diversify=True)
dt_div = (time.perf_counter() - t0) * 1000

df_div = pd.DataFrame([a.__dict__ for a in sol_div.assignments])
df_div

,parcela_id,variety,score,hard_passed
0,P1_Acatenango_alto,Catuai,0.45,"[altitud, ph, sombra]"
1,P2_Huehue_alto,Anacafe 14,0.80,"[altitud, ph, sombra]"
2,P3_SanMarcos_medio,Caturra,0.40,"[altitud, ph, sombra]"
3,P4_Coban_bajo,Catimor,0.80,"[altitud, ph, sombra]"
4,P5_FraijanesAlto,Bourbon,0.35,"[altitud, ph, sombra]"
5,P6_Antigua_medio,Marsellesa,0.80,"[altitud, ph, sombra]"


In [5]:
print(f'Score total:       {sol_div.total_score:.3f}')
print(f'Nodos expandidos:  {sol_div.nodes_expanded}')
print(f'Tiempo:            {dt_div:.2f} ms')
print(f'Variedades unicas: {df_div["variety"].nunique()} / {len(df_div)}')

Score total:       3.600
Nodos expandidos:  7551
Tiempo:            60.38 ms
Variedades unicas: 6 / 6


## 4. Resolución sin diversificación (`diversify=False`)

Cada parcela se asigna independientemente. Permite repetición — modelo monocultivo.

In [6]:
t0 = time.perf_counter()
sol_mono = solve(parcelas, varieties, diversify=False)
dt_mono = (time.perf_counter() - t0) * 1000

df_mono = pd.DataFrame([a.__dict__ for a in sol_mono.assignments])
df_mono

,parcela_id,variety,score,hard_passed
0,P1_Acatenango_alto,Anacafe 14,0.8,"[altitud, ph, sombra]"
1,P2_Huehue_alto,Anacafe 14,0.8,"[altitud, ph, sombra]"
2,P3_SanMarcos_medio,Catimor,0.8,"[altitud, ph, sombra]"
3,P4_Coban_bajo,Catimor,0.8,"[altitud, ph, sombra]"
4,P5_FraijanesAlto,Anacafe 14,0.8,"[altitud, ph, sombra]"
5,P6_Antigua_medio,Anacafe 14,0.8,"[altitud, ph, sombra]"


In [7]:
print(f'Score total:       {sol_mono.total_score:.3f}')
print(f'Nodos expandidos:  {sol_mono.nodes_expanded}')
print(f'Tiempo:            {dt_mono:.2f} ms')
print(f'Variedades unicas: {df_mono["variety"].nunique()} / {len(df_mono)}')

Score total:       4.800
Nodos expandidos:  22532
Tiempo:            123.91 ms
Variedades unicas: 2 / 6


## 5. Validación de restricciones duras

Verificación explícita: toda asignación devuelta debe cumplir altitud, pH y sombra de la parcela correspondiente. La tasa de satisfacción debe ser 100%.

In [8]:
variety_by_name = {v.name: v for v in varieties}
parcela_by_id = {p.parcela_id: p for p in parcelas}

def check_solution(sol):
    ok = 0
    for a in sol.assignments:
        p, v = parcela_by_id[a.parcela_id], variety_by_name[a.variety]
        if satisfies_hard_constraints(v, p.altitud, p.ph, p.sombra_disponible):
            ok += 1
    return ok, len(sol.assignments)

print('diversify=True ', check_solution(sol_div))
print('diversify=False', check_solution(sol_mono))

diversify=True  (6, 6)
diversify=False (6, 6)


## 6. Sensibilidad a pesos de soft score

Tres configuraciones: priorizar resistencia a roya, priorizar rendimiento, balanceado.

In [9]:
configs = [
    ('rust_priority',  1.0, 0.0),
    ('yield_priority', 0.0, 1.0),
    ('balanced',       0.5, 0.5),
]

rows = []
for label, wr, wy in configs:
    sol = solve(parcelas, varieties, diversify=True, w_rust=wr, w_yield=wy)
    for a in sol.assignments:
        rows.append({'config': label, 'parcela': a.parcela_id, 'variedad': a.variety, 'score': round(a.score, 3)})

pd.DataFrame(rows).pivot(index='parcela', columns='config', values='variedad')

config,balanced,rust_priority,yield_priority
parcela,,,
P1_Acatenango_alto,Catuai,Marsellesa,Marsellesa
P2_Huehue_alto,Anacafe 14,Anacafe 14,Anacafe 14
P3_SanMarcos_medio,Caturra,Caturra,Catuai
P4_Coban_bajo,Catimor,Catimor,Catimor
P5_FraijanesAlto,Bourbon,Bourbon,Bourbon
P6_Antigua_medio,Marsellesa,Catuai,Caturra


## 7. Caso infactible

Parcela con altitud fuera del rango de toda variedad: el solver levanta `ValueError`.

In [10]:
try:
    solve([Parcela('P_imposible', altitud=400, ph=5.0, sombra_disponible=1)], varieties)
except ValueError as e:
    print('ValueError ->', e)

ValueError -> Parcelas sin variedad compatible: ['P_imposible']
